In [0]:
%pip install confluent-kafka requests # Run Only Once

# Loading Libraries

In [0]:
import requests

import random
from datetime import datetime
import json
from confluent_kafka import Producer
from datetime import datetime, timezone

In [0]:
# Load all credentials from Databricks secrets
fred_api_key           = dbutils.secrets.get(scope='silk_pipeline', key='fred_api_key')
confluent_api_key      = dbutils.secrets.get(scope='silk_pipeline', key='confluent_api_key')
confluent_api_secret   = dbutils.secrets.get(scope='silk_pipeline', key='confluent_api_secret')
bootstrap_server       = dbutils.secrets.get(scope='silk_pipeline', key='confluent_bootstrap_server')

In [0]:


def fetch_fred_silk_price():
    response = requests.get(
        'https://api.stlouisfed.org/fred/series/observations',
        params={
            'series_id':  'WPU034203',  # Silk & Natural Fiber PPI
            'api_key':    fred_api_key,
            'file_type':  'json',
            'sort_order': 'desc',
            'limit':       1             # Latest value only
        }
    )
    data       = response.json()
    latest     = data['observations'][0]
    ppi_value  = float(latest['value'])
    ppi_date   = latest['date']
    return ppi_value, ppi_date

ppi_value, ppi_date = fetch_fred_silk_price()
print(f"Latest PPI: {ppi_value} as of {ppi_date}")

Latest PPI: 240.218 as of 2026-04-01


In [0]:


def simulate_weekly_price(ppi_value):
    # PPI of ~240 should map to ~$60,000/MT
    # 60000 / 240 = 250 scaling factor
    base_price   = ppi_value * 250
    fluctuation  = random.uniform(-0.015, 0.015)
    weekly_price = round(base_price * (1 + fluctuation), 2)
    return weekly_price

weekly_price = simulate_weekly_price(ppi_value)
print(f"Simulated weekly silk price: ${weekly_price}/MT")

Simulated weekly silk price: $59563.16/MT


In [0]:

# Kafka config
producer = Producer({
    'bootstrap.servers':       bootstrap_server,
    'security.protocol':       'SASL_SSL',
    'sasl.mechanism':          'PLAIN',
    'sasl.username':           confluent_api_key,
    'sasl.password':           confluent_api_secret
})

# Build message
message = {
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'ppi_date':    ppi_date,
    'ppi_value':   ppi_value,
    'price_usd_mt': weekly_price,
    'currency':    'USD',
    'unit':        'per MT',
    'source':      'FRED WPU034203 + simulation'
}

# Publish
producer.produce(
    topic     = 'silk-prices',
    key       = 'silk',
    value     = json.dumps(message)
)
producer.flush(timeout=30)  # Wait up to 30 seconds

print(f"Message published successfully!")
print(f"Payload: {json.dumps(message, indent=2)}")

Message published successfully!
Payload: {
  "timestamp": "2026-06-03T07:56:44.952772+00:00",
  "ppi_date": "2026-04-01",
  "ppi_value": 240.218,
  "price_usd_mt": 59563.16,
  "currency": "USD",
  "unit": "per MT",
  "source": "FRED WPU034203 + simulation"
}


%6|1780473406.173|GETSUBSCRIPTIONS|rdkafka#producer-6| [thrd:main]: Telemetry client instance id changed from AAAAAAAAAAAAAAAAAAAAAA to yc7M32FCSaqGRVIYd4KjAA
